# 01 — Simulation Pipeline

This notebook executes the full 13-phase governance friction simulation.
All computation is delegated to `scripts/run_all.py`; this notebook is orchestration only.

In [1]:
import os
import sys
from pathlib import Path

# Resolve repo root: NotebookClient sets cwd to the root, but interactive runs may start in notebooks/.
def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "scripts" / "run_all.py").is_file():
        return cwd
    if (cwd.parent / "scripts" / "run_all.py").is_file():
        return cwd.parent
    raise RuntimeError(
        "Cannot find repository root (missing scripts/run_all.py). cwd=%s" % cwd
    )


_root = _repo_root()
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Working directory:", _root)

Working directory: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-5-sensitivity-study


In [2]:
# Execute full simulation pipeline
from scripts.run_all import main
summary = main()

GOVERNANCE FRICTION SIMULATION v1.0
Analysis plan hash: 1fba2e2435e8fbf2...
SCM graph: 28 nodes, 19 edges
SCM is valid DAG: True

--- Phase 1: Validation Tests ---


Validation: 7/7 tests passed
All validation tests PASSED

--- Phase 2: Baseline Simulation ---
  Scenario 1/5: {'unsafe_base_rate': 0.1, 'auditability_noise': 0.05, 'evidence_regime': 'mixed', 'drift_regime': 'medium'}


  Scenario 2/5: {'unsafe_base_rate': 0.2, 'auditability_noise': 0.05, 'evidence_regime': 'mixed', 'drift_regime': 'medium'}


  Scenario 3/5: {'unsafe_base_rate': 0.3, 'auditability_noise': 0.05, 'evidence_regime': 'mixed', 'drift_regime': 'medium'}


  Scenario 4/5: {'unsafe_base_rate': 0.2, 'auditability_noise': 0.02, 'evidence_regime': 'artefact-heavy', 'drift_regime': 'low'}


...

  Mean final decay: 0.0326
  Re-audit rate: 0.101

--- Phase 7: Sobol Sensitivity Analysis ---


  unsafe_detection_rate: top driver = safety_gate (ST=1.000)
  safe_throughput: top driver = calibration_gate (ST=0.318)
  false_negative_harm: top driver = safety_gate (ST=1.000)
  mean_total_friction: top driver = safety_gate (ST=0.231)

--- Phase 8: Decision Curve Analysis ---


  DCA complete across 5 policies

--- Phase 9: Statistical Inference ---
  Detection: 0.996 [0.994, 0.998]
  Throughput: 0.210 [0.203, 0.217]

--- Phase 10: Replication Protocol ---


  Replication: PASS - All metrics within tolerance

--- Phase 11: Generating Figures ---


  Generated 6 figures

--- Phase 12: Generating Tables ---

--- Phase 13: Creating Run Manifest ---

SIMULATION COMPLETE
  Total evaluations: 100
  Pareto solutions: 60
  Validation: PASSED
  Replication: PASS - All metrics within tolerance
  Elapsed time: 361.7s
  Outputs saved to: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-5-sensitivity-study\outputs


In [3]:
# Display validation summary
import json
from pathlib import Path
val_report = json.loads(Path('outputs/logs/validation_report.json').read_text())
print(f"Validation: {val_report['n_passed']}/{val_report['n_tests']} tests passed")
for name, result in val_report['details'].items():
    status = 'PASS' if result['passed'] else 'FAIL'
    print(f"  {status}: {name}")

Validation: 7/7 tests passed
  PASS: safety_monotonicity
  PASS: bias_subgroup_harm
  PASS: calibration_abstention
  PASS: drift_behaviour
  PASS: gaming_behaviour
  PASS: observation_noise_reduction
  PASS: value_bounds


In [4]:
# Display replication summary
rep_report = json.loads(Path('reproducibility/replication_report.json').read_text())
print(f"Replication: {rep_report['summary']}")
for metric, vals in rep_report['metric_comparisons'].items():
    print(f"  {metric}: diff={vals['absolute_difference']}, passed={vals['passed']}")

Replication: PASS - All metrics within tolerance
  unsafe_detection_rate: diff=0.0, passed=True
  safe_throughput: diff=0.0, passed=True
  false_negative_harm: diff=0.0, passed=True
  mean_total_friction: diff=0.0, passed=True


In [5]:
# Display run manifest
manifest = json.loads(Path('reproducibility/run_manifest.json').read_text())
print(f"Plan hash: {manifest['plan_hash'][:16]}...")
print(f"Total evaluations: {manifest['total_evaluations']}")
print(f"Pareto solutions: {manifest['nsga2_pareto_solutions']}")
print(f"Validation passed: {manifest['validation_passed']}")
print(f"Replication passed: {manifest['replication_passed']}")
print(f"Elapsed: {manifest['elapsed_seconds']}s")

Plan hash: 1fba2e2435e8fbf2...
Total evaluations: 100
Pareto solutions: 60
Validation passed: True
Replication passed: True
Elapsed: 361.7s


In [6]:
# Verify all expected output files exist
from pathlib import Path
expected = [
    'outputs/raw/scenario_results.csv',
    'outputs/raw/compensatory_comparison.csv',
    'outputs/processed/grid_results.json',
    'outputs/processed/pareto_solutions.csv',
    'outputs/processed/dca_results.json',
    'outputs/processed/inference_results.json',
    'outputs/processed/simulation_summary.json',
    'outputs/tables/table1_thresholds.csv',
    'outputs/tables/table2_pareto.csv',
    'outputs/figures/fig1_frontier.png',
    'outputs/figures/fig2_heatmap.png',
    'outputs/figures/fig3_sensitivity.png',
    'outputs/figures/fig4_drift.png',
    'outputs/figures/fig5_dca.png',
    'outputs/figures/fig6_scm.png',
]
for f in expected:
    assert Path(f).exists(), f'Missing: {f}'
print(f'All {len(expected)} expected output files present.')

All 15 expected output files present.
